In [ ]:
!pip install ipywidgets numpy pillow

In [ ]:
import numpy as np
from PIL import Image
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display

# ── Resolution ──────────────────────────────────────────────────────────────
WIDTH, HEIGHT = 320, 240          # internal render resolution
DISPLAY_WIDTH  = 640
DISPLAY_HEIGHT = 480

# ── Fixed-point parameters ───────────────────────────────────────────────────
MAX_ITER  = 30
SCALE     = 4096
FRAC_BITS = 12

RE_MIN, RE_MAX = -2.0,  2.0
IM_MIN, IM_MAX = -1.5,  1.5

DEFAULT_STEP = int(round((RE_MAX - RE_MIN) * SCALE / (WIDTH - 1)))
DEFAULT_ZR0  = int(round(RE_MIN * SCALE))
DEFAULT_ZI0  = int(round(IM_MIN * SCALE))

ROOTS_F = [
    ( 1.0,        0.0      ),
    (-0.5,        0.8660254),
    (-0.5,       -0.8660254)
]
ROOTS = [(int(round(r * SCALE)), int(round(i * SCALE))) for r, i in ROOTS_F]

COL = np.array([
    [230,  57,  70],
    [ 42, 157, 143],
    [ 69, 123, 157]
], dtype=int)

TOL       = int(round(0.03 * SCALE))
DENOM_MIN = 1

# ── Fixed-point arithmetic ───────────────────────────────────────────────────
def tdiv(a, b):
    if b == 0:
        return 0
    q = abs(a) // abs(b)
    return -q if (a < 0) != (b < 0) else q

def mul(a, b):
    return tdiv(a * b, SCALE)

# ── View helpers ─────────────────────────────────────────────────────────────
def view_to_registers(zoom_value=1.0, pan_x_value=0, pan_y_value=0):
    step = max(1, int(round(DEFAULT_STEP / zoom_value)))
    centre_r = DEFAULT_ZR0 + (WIDTH  // 2) * DEFAULT_STEP + pan_x_value * step
    centre_i = DEFAULT_ZI0 + (HEIGHT // 2) * DEFAULT_STEP + pan_y_value * step
    zr0 = centre_r - (WIDTH  // 2) * step
    zi0 = centre_i - (HEIGHT // 2) * step
    return int(zr0), int(zi0), int(step)

# ── Fractal renderer ─────────────────────────────────────────────────────────
def generate_image(zoom_value=1.0, pan_x_value=0, pan_y_value=0, max_iter=MAX_ITER):
    zr0, zi0, step = view_to_registers(zoom_value, pan_x_value, pan_y_value)
    img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    for py in range(HEIGHT):
        for px in range(WIDTH):
            zr = zr0 + px * step
            zi = zi0 + py * step
            which = -1
            it    = 0

            for it in range(max_iter):
                zr2 = mul(zr, zr) - mul(zi, zi)
                zi2 = tdiv(2 * zr * zi, SCALE)
                zr3 = mul(zr2, zr) - mul(zi2, zi)
                zi3 = mul(zr2, zi) + mul(zi2, zr)
                fr  = zr3 - SCALE
                fi  = zi3
                fpr = 3 * zr2
                fpi = 3 * zi2
                denom = mul(fpr, fpr) + mul(fpi, fpi)
                if denom <= DENOM_MIN:
                    break
                numr = mul(fr, fpr) + mul(fi, fpi)
                numi = mul(fi, fpr) - mul(fr, fpi)
                dr   = tdiv(numr * SCALE, denom)
                di   = tdiv(numi * SCALE, denom)
                zr  -= dr
                zi  -= di
                for k, (rr, ri) in enumerate(ROOTS):
                    if abs(zr - rr) < TOL and abs(zi - ri) < TOL:
                        which = k
                        break
                if which >= 0:
                    break

            if which < 0:
                img[py, px] = (0, 0, 0)
            else:
                shade256 = max(64, 256 - (it * 256) // max_iter)
                img[py, px] = (
                    (int(COL[which][0]) * shade256) >> 8,
                    (int(COL[which][1]) * shade256) >> 8,
                    (int(COL[which][2]) * shade256) >> 8
                )
    return img

def to_png(img_array):
    buf = BytesIO()
    Image.fromarray(np.asarray(img_array, dtype=np.uint8)).save(buf, format='PNG')
    return buf.getvalue()

# ── Widgets ──────────────────────────────────────────────────────────────────
image_widget = widgets.Image(value=b'', format='png',
                             width=DISPLAY_WIDTH, height=DISPLAY_HEIGHT)

zoom_slider  = widgets.FloatSlider(value=1.0, min=0.5, max=20.0, step=0.1,
                                   description='Zoom',     continuous_update=False)
pan_x_slider = widgets.IntSlider(  value=0,   min=-320, max=320,  step=1,
                                   description='Pan X',    continuous_update=False)
pan_y_slider = widgets.IntSlider(  value=0,   min=-240, max=240,  step=1,
                                   description='Pan Y',    continuous_update=False)
maxit_slider = widgets.IntSlider(  value=MAX_ITER, min=1, max=63, step=1,
                                   description='Max Iter', continuous_update=False)

apply_button = widgets.Button(description='Apply View')
reset_button = widgets.Button(description='Reset View')
status       = widgets.Label(value='Press Apply View to render.')

# ── Render ───────────────────────────────────────────────────────────────────
def render(zoom_value, pan_x_value, pan_y_value, max_iter):
    status.value = 'Rendering...'
    img = generate_image(zoom_value, pan_x_value, pan_y_value, max_iter)
    image_widget.value = to_png(img)
    zr0, zi0, step = view_to_registers(zoom_value, pan_x_value, pan_y_value)
    status.value = (f'Zoom={zoom_value:.2f}  Pan X={pan_x_value}  Pan Y={pan_y_value}  '
                    f'Max Iter={max_iter}  |  ZR0={zr0}  ZI0={zi0}  STEP={step}')

def apply_view(btn):
    render(zoom_slider.value, pan_x_slider.value, pan_y_slider.value, maxit_slider.value)

def reset_view(btn):
    zoom_slider.value  = 1.0
    pan_x_slider.value = 0
    pan_y_slider.value = 0
    maxit_slider.value = MAX_ITER
    render(1.0, 0, 0, MAX_ITER)

apply_button.on_click(apply_view)
reset_button.on_click(reset_view)

# ── Layout ───────────────────────────────────────────────────────────────────
display(
    image_widget,
    zoom_slider,
    pan_x_slider,
    pan_y_slider,
    maxit_slider,
    widgets.HBox([apply_button, reset_button]),
    status
)

# Render default view on load
render(1.0, 0, 0, MAX_ITER)